# Live collector + dual-listing trader (one shared `Exchange`)

Connect **once** to Optibook, then each outer-loop cycle:

1. **`collector_poll_once`** — append one CSV row per instrument (same as one iteration inside `run_collector`).
2. **`Trader.step`** — one trading tick (same logic as inside `Trader.run`, without the inner sleep).

Optibook allows **only one active session** per account; this notebook holds that session for both tasks.

Run from **repo root** with `PYTHONPATH=.` (or set `REPO` below and extend `sys.path`).

In [ ]:
from pathlib import Path
import sys
import time

REPO = None
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "trader.py").is_file() and (_p / "data" / "collect_strategy_data.py").is_file():
        REPO = _p
        break
if REPO is None:
    raise RuntimeError("Run from repo root (need trader.py and data/collect_strategy_data.py)")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("REPO", REPO)

In [ ]:
from optibook.synchronous_client import Exchange

from data.collect_strategy_data import (
    collector_poll_once,
    init_collector_state,
)
from trader import Trader

# Seconds to sleep after each full cycle (collector + trader step).
CYCLE_SLEEP = 1.0

In [ ]:
exchange = Exchange()
exchange.connect()
print("connected", exchange.is_connected())

In [ ]:
# One-time CSV universe (writes under data/csv/ by default)
universe, paths = init_collector_state(exchange, output_dir=REPO / "data" / "csv")

trader = Trader()
trader.start(exchange)
print("collector instruments:", len(universe), "dual pairs:", Trader.DUAL_PAIRS)

In [ ]:
# Run until Ctrl+C or disconnect
try:
    n = 0
    while True:
        if not exchange.is_connected():
            print("Exchange disconnected; stopping loop.")
            break

        ok = collector_poll_once(exchange, universe, paths)
        if not ok:
            break

        trader.step(exchange)

        n += 1
        if n % 30 == 0:
            print(f"cycles={n}  connected={exchange.is_connected()}")

        time.sleep(CYCLE_SLEEP)
except KeyboardInterrupt:
    print("Stopped by user (KeyboardInterrupt).")